# End-sem 04. The Shadow Squad engine
Style translation and its check on real movers, the candidate scores, and the learning-to-rank fusion.

In [1]:
import sys, os; sys.path.insert(0, 'src')
import pandas as pd, numpy as np, pickle, time
from scipy import stats
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from pyspark.sql import functions as F, Window as W
from common import *
import engine as E
sp = spark(); RT, M = {}, {}
def savefig(name): plt.tight_layout(); plt.savefig(f'outputs/figs/{name}.png', dpi=160); plt.close()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/14 00:42:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1. Market value at t and the hazard score (inputs the engine reads from the lake)
Value = last Transfermarkt valuation dated on or before 15 June after season t. Hazard = discrete-time survival model: Spark logistic regression on person-period rows, refitted for every season t on rows whose 12-month outcome was already known (seasons up to t-1).

In [2]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from pyspark.ml.functions import vector_to_array
with timed('value at t', RT):
    ft = sp.read.parquet('lake/features').select('player_id', 'yr')
    v = sp.read.parquet('lake/tm_valuations').select('player_id', 'date', F.col('market_value_in_eur').cast('double').alias('value'))
    val = (ft.join(v, (ft.player_id == v.player_id) & (v.date <= F.make_date(ft.yr + 1, F.lit(6), F.lit(15)))).select(ft.player_id, 'yr', 'date', 'value')
             .withColumn('r', F.row_number().over(W.partitionBy('player_id', 'yr').orderBy(F.desc('date')))).filter('r = 1').drop('r', 'date'))
    val.toPandas().to_parquet('lake/value_at_t.parquet', index=False)
lake = E.Lake(); FT = lake.F
HF = E.ZC + ['dur', 'age_num', 'composite_perf_index', 'min_num'] + ['P_' + p for p in sorted(FT['fine_pos'].unique())] + ['L_' + str(i) for i in range(1, 5)]
X = FT[['player_id', 'yr', 'fine_pos', 'comp', 'dur', 'age_num', 'composite_perf_index', 'min_num'] + E.ZC].copy()
for p in sorted(FT['fine_pos'].unique()): X['P_' + p] = (X['fine_pos'] == p).astype(float)
for i, L_ in enumerate(LEAGUES[1:], start=1): X['L_' + str(i)] = (X['comp'] == L_).astype(float)
PP = pd.read_parquet('lake/person_period.parquet')[['player_id', 'yr', 'event']]
xs = sp.createDataFrame(X[['player_id', 'yr'] + HF].fillna(0)).cache(); pp = sp.createDataFrame(PP).join(xs, ['player_id', 'yr']).cache()
pipe = lambda: Pipeline(stages=[VectorAssembler(inputCols=HF, outputCol='raw'), StandardScaler(inputCol='raw', outputCol='features', withMean=True), LogisticRegression(labelCol='event', regParam=0.05)])
hz = []
with timed('discrete-time hazard, Spark LR, 6 refits', RT):
    for t in range(2018, 2024):
        mdl = pipe().fit(pp.filter(F.col('yr') <= t - 1)); hz.append(mdl.transform(xs.filter(F.col('yr') == t)).select('player_id', 'yr', vector_to_array('probability')[1].alias('hazard')).toPandas())
mdl.transform(xs).select('probability').explain()
HZ = pd.concat(hz); HZ.to_parquet('lake/hazard.parquet', index=False); lake = E.Lake(); FT = lake.F
print(f"value known for {lake.VAL.notna().sum():,} of {len(FT):,} player-seasons; hazard scored for {len(HZ):,} player-seasons (2018-2023; no earlier season to learn from for 2017)")

[value at t] 3.6 s


netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory


[discrete-time hazard, Spark LR, 6 refits] 14.0 s
== Physical Plan ==
*(1) Project [UDF(UDF(UDF(raw#28116))) AS probability#28130]
+- *(1) Project [UDF(struct(z_gls, z_gls#114, z_npxg, z_npxg#115, z_sh, z_sh#116, z_ast, z_ast#117, z_kp, z_kp#118, z_sca, z_sca#119, z_ppa, z_ppa#120, z_prgp, z_prgp#121, z_prgc, z_prgc#122, z_take_on, z_take_on#123, z_att_pen, z_att_pen#124, z_tklw, z_tklw#125, z_int, ... 37 more fields)) AS raw#28116]
   +- InMemoryTableScan [L_1#141, L_2#142, L_3#143, L_4#144, P_AM#134, P_CB#135, P_CM#136, P_DM#137, P_FB#138, P_ST#139, P_W#140, age_num#131, composite_perf_index#132, dur#130L, min_num#133L, z_aer_won#129, z_ast#117, z_att_pen#124, z_blocks#128, z_clr#127, z_gls#114, z_int#126, z_kp#118, z_npxg#115, z_ppa#120, ... 6 more fields]
         +- InMemoryRelation [player_id#112, yr#113L, z_gls#114, z_npxg#115, z_sh#116, z_ast#117, z_kp#118, z_sca#119, z_ppa#120, z_prgp#121, z_prgc#122, z_take_on#123, z_att_pen#124, z_tklw#125, z_int#126, z_clr#127, z_blocks#128

value known for 10,266 of 10,267 player-seasons; hazard scored for 8,787 player-seasons (2018-2023; no earlier season to learn from for 2017)


## 2. Style translation, checked on players who really changed club
For every player with >= 900 minutes in two consecutive seasons at different clubs, his per-90 numbers at the new club are predicted three ways: (a) last season's raw numbers unchanged, (b) his style-neutral numbers, (c) his neutral numbers translated into the new club. The new club's style is taken from the season **before** he arrives, which is what a scout knows; clubs with no top-flight row that season get the neutral value.

In [3]:
A = FT.copy(); B = FT[['player_id', 'yr', 'squad', 'comp', 'x_dom', 'x_press', 'pos_group'] + E.RAW].copy(); B['yr'] -= 1
mvr = A.merge(B, on=['player_id', 'yr'], suffixes=('', '_n')); mvr = mvr[(mvr['squad'] != mvr['squad_n']) & (mvr['pos_group'] == mvr['pos_group_n'])].reset_index(drop=True)
prev = FT.groupby(['comp', 'squad', 'yr'])[['x_dom', 'x_press']].first(); known = [prev.index.isin([k])[0] if False else (k in prev.index) for k in zip(mvr['comp_n'], mvr['squad_n'], mvr['yr'])]
xd = np.array([prev.loc[k, 'x_dom'] if ok else 0.0 for k, ok in zip(zip(mvr['comp_n'], mvr['squad_n'], mvr['yr']), known)]); xp = np.array([prev.loc[k, 'x_press'] if ok else 0.0 for k, ok in zip(zip(mvr['comp_n'], mvr['squad_n'], mvr['yr']), known)])
act = mvr[[r + '_n' for r in E.RAW]].to_numpy(); pa = mvr[E.RAW].to_numpy(); pb = mvr[E.AF].to_numpy()
pc = lake.translate(pb, mvr['pos_group'].to_numpy(), xd, xp, mvr['comp_n'].to_numpy()); pc_or = lake.translate(pb, mvr['pos_group'].to_numpy(), mvr['x_dom_n'].to_numpy(), mvr['x_press_n'].to_numpy(), mvr['comp_n'].to_numpy())
scale = FT[E.RAW].std().to_numpy(); err = {k: np.abs(p - act) / scale for k, p in (('a_raw', pa), ('b_neutral', pb), ('c_translated', pc), ('c_oracle_style', pc_or))}
tabs = []
for g in ('DF', 'MF', 'FW', 'all'):
    msk = np.ones(len(mvr), bool) if g == 'all' else (mvr['pos_group'] == g).to_numpy()
    for i, m in enumerate(E.MET): tabs.append({'pos': g, 'metric': m, 'n': int(msk.sum()), **{k: float(e[msk, i].mean()) for k, e in err.items()}})
TV = pd.DataFrame(tabs); TV['c_beats_a'] = TV['c_translated'] < TV['a_raw']; TV.to_csv('outputs/translation_validation.csv', index=False)
rng = np.random.default_rng(SEED); bs = rng.integers(0, len(mvr), (2000, len(mvr))); pe = {k: e.mean(1) for k, e in err.items()}
ci = lambda d: [float(x) for x in np.percentile(d[bs].mean(1), [2.5, 97.5])]
cross = (mvr['comp'] != mvr['comp_n']).to_numpy(); big = (np.abs(xd - mvr['x_dom'].to_numpy()) > 0.5)
M['translation'] = {'movers': len(mvr), 'cross_league_movers': int(cross.sum()), 'new_club_style_known': int(sum(known)), 'mean_scaled_mae': {k: float(v.mean()) for k, v in pe.items()},
    'c_minus_a_ci95': ci(pe['c_translated'] - pe['a_raw']), 'c_minus_b_ci95': ci(pe['c_translated'] - pe['b_neutral']), 'b_minus_a_ci95': ci(pe['b_neutral'] - pe['a_raw']), 'oracle_minus_a_ci95': ci(pe['c_oracle_style'] - pe['a_raw']),
    'cells_c_beats_a': int(TV[TV['pos'] != 'all']['c_beats_a'].sum()), 'cells_total': int((TV['pos'] != 'all').sum()), 'metrics_c_beats_a_all_positions': TV[(TV['pos'] == 'all') & TV['c_beats_a']]['metric'].tolist(),
    'cross_league_only': {k: float(v[cross].mean()) for k, v in pe.items()}, 'big_style_change_only': {'n': int(big.sum()), **{k: float(v[big].mean()) for k, v in pe.items()}},
    'by_pos': {g: {k: float(TV[TV['pos'] == g][k].mean()) for k in err} for g in ('DF', 'MF', 'FW')}}
print({k: v for k, v in M['translation'].items()})
piv = TV[TV['pos'] == 'all'].set_index('metric'); fig, ax = plt.subplots(figsize=(8, 3.2)); x = np.arange(len(piv))
for j, (k, lab_) in enumerate((('a_raw', '(a) raw, unchanged'), ('b_neutral', '(b) style-neutral'), ('c_translated', '(c) translated into the new club'))): ax.bar(x + (j - 1) * .27, piv[k], .27, label=lab_)
ax.set_xticks(x, piv.index, rotation=60, fontsize=7); ax.set_ylabel('MAE at the new club (in SD of the metric)'); ax.legend(fontsize=7); ax.set_title(f'Predicting a mover\'s next-season per-90s ({len(mvr)} real club changes)', fontsize=9); savefig('p4_translation')

{'movers': 1110, 'cross_league_movers': 460, 'new_club_style_known': 953, 'mean_scaled_mae': {'a_raw': 0.4949362816115702, 'b_neutral': 0.4863147211874726, 'c_translated': 0.488755130051894, 'c_oracle_style': 0.4835847412497543}, 'c_minus_a_ci95': [-0.009624480988460466, -0.0027476351399800284], 'c_minus_b_ci95': [-0.0010316900978067942, 0.00608322169000389], 'b_minus_a_ci95': [-0.012464396312945025, -0.005132995175314797], 'oracle_minus_a_ci95': [-0.015032038340834624, -0.007885481243576162], 'cells_c_beats_a': 37, 'cells_total': 48, 'metrics_c_beats_a_all_positions': ['gls', 'npxg', 'sh', 'ast', 'kp', 'sca', 'prgp', 'take_on', 'int', 'clr', 'blocks', 'aer_won'], 'cross_league_only': {'a_raw': 0.529354191216598, 'b_neutral': 0.5054628023416823, 'c_translated': 0.5073562301015337, 'c_oracle_style': 0.5002816669142524}, 'big_style_change_only': {'n': 399, 'a_raw': 0.5123341421304611, 'b_neutral': 0.5050376162741862, 'c_translated': 0.5132564862834144, 'c_oracle_style': 0.505809186077396

## 3. Candidate scores for every replacement event
One row per (event, same-position pool player). `sim_translated` compares the candidate translated into the buying club with the departed starter's actual numbers there. `traj_sim` compares the last seasons, age-aligned where the starter's seasons at those ages are in the data.

In [4]:
EV = pd.read_parquet('lake/replacement_events'); EV['qid'] = EV['C'].astype(str) + '_' + EV['t'].astype(str) + '_' + EV['S'].astype(str)
Q = EV[['qid', 'C', 't', 'S', 'P', 'club', 'S_name']].drop_duplicates('qid')
with timed('pair features', RT): PF = lake.pair_features(Q)
CA = pd.read_parquet('lake/candidates.parquet')[['qid', 'player_id', 'dist', 'candidate', 'in_brp', 'in_minhash']]; PF = PF.merge(CA, on=['qid', 'player_id'], how='left'); PF['candidate'] = PF['candidate'].fillna(False).astype(bool)
pos_ = EV[EV['N_in_pool']][['qid', 'N']].drop_duplicates().assign(y=1).rename(columns={'N': 'player_id'}); PF = PF.merge(pos_, on=['qid', 'player_id'], how='left'); PF['y'] = PF['y'].fillna(0).astype(int)
PF['q_in_pool'] = PF.groupby('qid')['y'].transform('max') == 1; PF.to_parquet('lake/pairs.parquet', index=False)
M['pairs'] = {'rows': len(PF), 'queries': int(PF['qid'].nunique()), 'in_pool_queries': int(PF[PF['q_in_pool']]['qid'].nunique()), 'positives': int(PF['y'].sum()), 'positives_in_candidate_set': int(PF[PF['candidate']]['y'].sum()),
              'traj_available': float(PF['traj_sim'].notna().mean()), 'traj_age_aligned_share': float(PF['traj_age_aligned'].mean()), 'value_missing': float(PF['value'].isna().mean()), 'hazard_missing': float(PF['hazard'].isna().mean()),
              'mean_feature_pos_vs_neg': {f: [float(PF[PF['q_in_pool'] & (PF['y'] == 1)][f].mean()), float(PF[PF['q_in_pool'] & (PF['y'] == 0)][f].mean())] for f in E.FEATS + ['sim_raw', 'sim_neutral']}}
print({k: v for k, v in M['pairs'].items() if k != 'mean_feature_pos_vs_neg'}); display(pd.DataFrame(M['pairs']['mean_feature_pos_vs_neg'], index=['real signing', 'other pool players']).T.round(3))

[pair features] 2.4 s
{'rows': 32776, 'queries': 342, 'in_pool_queries': 95, 'positives': 102, 'positives_in_candidate_set': 97, 'traj_available': 1.0, 'traj_age_aligned_share': 0.13604466682938735, 'value_missing': 0.0001525506468147425, 'hazard_missing': 0.08158408591652429}


,real signing,other pool players
sim_translated,0.116,0.032
traj_sim,0.062,0.006
pipeline_rank,-0.150,-0.245
club_taste,0.621,0.566
hazard,0.172,0.109
log_value,7.134,6.844
value_gap,0.360,0.531
age_num,22.186,23.220
composite_perf_index,52.546,48.808
sim_raw,0.080,-0.012


### Trajectory: LSH on concatenated season vectors (Spark) and DTW as the out-of-course comparison
Players with two consecutive seasons get a 32-dimensional vector [z(t), z(t-1)]; Euclidean LSH finds near trajectories. Dynamic time warping, which allows seasons to stretch, is run on the in-pool events and compared with the simple aligned cosine.

In [5]:
from pyspark.ml.feature import BucketedRandomProjectionLSH
from pyspark.ml.linalg import Vectors, VectorUDT
from dtaidistance import dtw_ndim
two = FT.merge(FT[['player_id', 'yr'] + E.ZC].assign(yr=lambda d: d['yr'] + 1), on=['player_id', 'yr'], suffixes=('', '_prev')); cols2 = E.ZC + [c + '_prev' for c in E.ZC]
tw = VectorAssembler(inputCols=cols2, outputCol='v').transform(sp.createDataFrame(two[['player_id', 'yr', 'fine_pos'] + cols2])).select('player_id', 'yr', 'fine_pos', 'v').cache()
with timed('trajectory LSH self-join', RT):
    lm = BucketedRandomProjectionLSH(inputCol='v', outputCol='h', bucketLength=4.0, numHashTables=6, seed=SEED).fit(tw)
    lj = lm.approxSimilarityJoin(tw, tw, 1e6, 'd').filter('datasetA.fine_pos = datasetB.fine_pos and datasetA.player_id != datasetB.player_id').select(F.col('datasetA.player_id').alias('a'), F.col('datasetA.yr').alias('ay'), F.col('datasetB.player_id').alias('b'), F.col('datasetB.yr').alias('by'), 'd')
    n_lsh = lj.count(); near = lj.withColumn('r', F.row_number().over(W.partitionBy('a', 'ay').orderBy('d'))).filter('r <= 10').count()
V2 = two[cols2].to_numpy(); fp2 = two['fine_pos'].to_numpy(); rs = rng.choice(len(two), 300, replace=False); hit = tot = 0; pairs_exact = sum(int((fp2 == p).sum()) ** 2 for p in np.unique(fp2))
LJ = lj.toPandas(); lset = set(zip(LJ['a'], LJ['ay'], LJ['b'], LJ['by']))
for i in rs:
    same = np.flatnonzero((fp2 == fp2[i]) & (two['player_id'].to_numpy() != two['player_id'].iat[i])); nn = same[np.argsort(np.linalg.norm(V2[same] - V2[i], axis=1))[:10]]
    hit += sum((two['player_id'].iat[i], two['yr'].iat[i], two['player_id'].iat[j], two['yr'].iat[j]) in lset for j in nn); tot += len(nn)
seq = lambda pid, t: np.array([lake.Z[lake.idx[(pid, t - k)]] for k in (2, 1, 0) if (pid, t - k) in lake.idx])
IP = PF[PF['q_in_pool']].copy()
with timed('DTW on in-pool events', RT): IP['dtw'] = [dtw_ndim.distance(seq(a, t), seq(b, t)) / max(len(seq(a, t)), len(seq(b, t))) for a, b, t in zip(IP['player_id'], IP['S'], IP['t'])]
rk = lambda col, asc: IP.assign(r=IP.groupby('qid')[col].rank(ascending=asc, method='average'))[IP['y'] == 1]['r']
M['trajectory'] = {'two_season_vectors': len(two), 'lsh_pairs': int(n_lsh), 'lsh_pair_share': float(n_lsh / pairs_exact), 'lsh_recall_top10': float(hit / tot), 'spearman_traj_vs_dtw': float(stats.spearmanr(IP['traj_sim'], -IP['dtw'], nan_policy='omit')[0]),
                   'median_rank_true_signing_traj_sim': float(rk('traj_sim', False).median()), 'median_rank_true_signing_dtw': float(rk('dtw', True).median()), 'recall10_traj_sim': float((rk('traj_sim', False) <= 10).mean()), 'recall10_dtw': float((rk('dtw', True) <= 10).mean())}
print(M['trajectory'])

[trajectory LSH self-join] 19.9 s


[DTW on in-pool events] 0.2 s
{'two_season_vectors': 5884, 'lsh_pairs': 5401406, 'lsh_pair_share': 0.9646046525416252, 'lsh_recall_top10': 0.999, 'spearman_traj_vs_dtw': 0.42879192215283013, 'median_rank_true_signing_traj_sim': 40.5, 'median_rank_true_signing_dtw': 40.0, 'recall10_traj_sim': 0.16666666666666666, 'recall10_dtw': 0.16666666666666666}


## 4. Fusion: learning to rank
One query per replacement event, label 1 for the player actually signed, 0 for the other LSH candidates. Walk-forward: the model that ranks events of season Y is trained on events up to Y-1. In-course: Spark `GBTClassifier` (pointwise). Out of course: LightGBM `LGBMRanker` (LambdaMART), which optimises the order inside each query.

In [6]:
from pyspark.ml.classification import GBTClassifier
TEST = (2020, 2021, 2022, 2023)
with timed('LGBMRanker walk-forward', RT): LG, models = E.walk_forward(PF, E.FEATS, TEST, keep_models=True)
allq = PF[PF['candidate'] & PF['q_in_pool']]; models['all'] = E.fit_ranker(allq, E.FEATS); pickle.dump(models, open('lake/models.pkl', 'wb'))
sdf = sp.createDataFrame(PF[['qid', 'player_id', 't', 'y', 'candidate', 'q_in_pool'] + E.FEATS].fillna(-1.0)).cache(); gb = []
with timed('Spark GBTClassifier walk-forward', RT):
    for Y in TEST:
        tr = sdf.filter((F.col('t') <= Y - 1) & F.col('candidate') & F.col('q_in_pool')); pr_ = tr.agg(F.mean('y')).first()[0]; tr = tr.withColumn('w', F.when(F.col('y') == 1, 0.5 / pr_).otherwise(0.5 / (1 - pr_)))
        gm = Pipeline(stages=[VectorAssembler(inputCols=E.FEATS, outputCol='features'), GBTClassifier(labelCol='y', weightCol='w', maxIter=60, maxDepth=3, stepSize=0.1, seed=SEED)]).fit(tr)
        gb.append(gm.transform(sdf.filter(F.col('t') == Y)).select('qid', 'player_id', 't', 'y', 'candidate', vector_to_array('probability')[1].alias('score')).toPandas())
gm.transform(sdf).select('probability').explain()
GB = pd.concat(gb); GB['score'] = np.where(GB['candidate'], GB['score'], -1e9)
def quick(R):
    R = R[R.groupby('qid')['y'].transform('max') == 1].copy(); R['r'] = R.groupby('qid')['score'].rank(ascending=False, method='average'); r = R[R['y'] == 1].groupby('qid')['r'].min()
    return {'queries': len(r), 'median_rank': float(r.median()), **{f'recall@{k}': float((r <= k).mean()) for k in (5, 10, 25)}}
LG.to_parquet('lake/scores_lgbm.parquet', index=False); GB.drop(columns='candidate').to_parquet('lake/scores_gbt.parquet', index=False)
imp = pd.Series(models['all'].booster_.feature_importance('gain'), index=E.FEATS); imp = (imp / imp.sum()).sort_values(ascending=False)
M['fusion'] = {'test_seasons': list(TEST), 'train_queries_first_fold': int(PF[(PF['t'] <= TEST[0] - 1) & PF['q_in_pool']]['qid'].nunique()), 'lgbm': quick(LG), 'gbt_spark': quick(GB), 'gain_share_all_events_model': imp.round(4).to_dict(), 'features': E.FEATS}
print(M['fusion'])
ex = EV[(EV['N_in_pool']) & (EV['t'] >= 2020)].iloc[0]; top = E.rank(lake, models, ex['C'], ex['t'], ex['S'], ex['P'], top=10)
print(f"example: {ex['club']} {ex['t']}-{str(ex['t'] + 1)[2:]}, replacing {ex['S_name']} ({ex['P']}); real signing {ex['N_name']}; model {top['model_used'][0]}"); display(top[['player', 'squad', 'age_num', 'value', 'sim_translated', 'traj_sim', 'hazard', 'score']].round(3))
_ = put_metrics('phase4', {**M, 'runtime_s': RT})

[LGBMRanker walk-forward] 0.5 s


[Spark GBTClassifier walk-forward] 53.8 s
== Physical Plan ==
*(1) Project [UDF(UDF(features#35072)) AS probability#35079]
+- *(1) Project [UDF(struct(sim_translated, sim_translated#29403, traj_sim, traj_sim#29404, pipeline_rank, pipeline_rank#29405, club_taste, club_taste#29406, hazard, hazard#29407, log_value, log_value#29408, value_gap, value_gap#29409, age_num, age_num#29410, composite_perf_index, composite_perf_index#29411)) AS features#35072]
   +- InMemoryTableScan [age_num#29410, club_taste#29406, composite_perf_index#29411, hazard#29407, log_value#29408, pipeline_rank#29405, sim_translated#29403, traj_sim#29404, value_gap#29409]
         +- InMemoryRelation [qid#29397, player_id#29398, t#29399L, y#29400L, candidate#29401, q_in_pool#29402, sim_translated#29403, traj_sim#29404, pipeline_rank#29405, club_taste#29406, hazard#29407, log_value#29408, value_gap#29409, age_num#29410, composite_perf_index#29411], StorageLevel(disk, memory, deserialized, 1 replicas)
               +- Lo

,player,squad,age_num,value,sim_translated,traj_sim,hazard,score
0,Erhan Mašović,Bochum,23.0,5000000.0,0.397,0.309,0.088,0.808
1,Ismaël Doukouré,Strasbourg,19.0,3000000.0,0.477,0.419,0.201,0.755
2,Marc Guéhi,Crystal Palace,22.0,35000000.0,0.120,-0.141,0.145,0.419
3,Gastón Álvarez,Getafe,22.0,5000000.0,0.517,0.466,0.088,0.368
4,Jean-Clair Todibo,Nice,22.0,30000000.0,0.050,0.169,0.161,0.346
5,Castello Lukeba,Lyon,19.0,20000000.0,0.162,-0.044,0.294,0.074
6,Unai Núñez,Celta Vigo,25.0,6000000.0,0.093,-0.268,0.152,-0.035
7,Marco Friedl,Werder Bremen,24.0,6500000.0,-0.013,0.141,0.059,-0.043
8,Anthony Rouault,Toulouse,21.0,8000000.0,0.327,0.123,0.213,-0.168
9,Daniel Vivian,Athletic Club,23.0,14000000.0,0.315,0.160,0.091,-0.217
